In [1]:
import pandas as pd

In [2]:
import ee
import geemap

# ---- 1. Authenticate (first run opens a browser — sign in once) ----
ee.Authenticate()
ee.Initialize(project='my-first-project1-498804')   # <-- your GEE project id, see note below

# ---- 2. Load YOUR shapefile directly ----
shp_path = r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp"
lahore_fc = geemap.shp_to_ee(shp_path)
lahore = lahore_fc.geometry()

# ---- 3. VIIRS night-time lights: 2025 median, clipped to your boundary ----
ntl = (ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
       .filterDate('2025-01-01', '2026-01-01')
       .select('avg_rad')
       .median()
       .clip(lahore))

# ---- 4. Interactive map (shows in notebook) ----
Map = geemap.Map()
Map.centerObject(lahore, 10)
vis = {'min': 0, 'max': 60,
       'palette': ['000000', '3a2c5f', 'd97706', 'ffd166', 'ffffff']}
Map.addLayer(ntl, vis, 'NTL 2025')
Map.addLayer(ee.Image().paint(lahore, 1, 2), {'palette': ['E3A93C']}, 'District boundary')
Map

# ---- 5. Total radiance inside your district (number for your paper) ----
stats = ntl.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=lahore,
    scale=500,
    maxPixels=1e9
).getInfo()
print('Total radiance inside Lahore District:', stats)

# ---- 6. Download GeoTIFF straight to your PC ----
out_tif = r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif"
geemap.ee_export_image(
    ntl,
    filename=out_tif,
    scale=500,
    region=lahore,
    crs='EPSG:4326',
    file_per_band=False
)
print('Saved:', out_tif)

Total radiance inside Lahore District: {'avg_rad': 94585.8068709108}
Generating URL ...
Please wait ...
Data downloaded to E:\economic_wealth_dashboard\lahore_ntl_2025.tif
Saved: E:\economic_wealth_dashboard\lahore_ntl_2025.tif


In [3]:
import rasterio, json

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    tr = src.transform

cells = []
for r in range(a.shape[0]):
    for c in range(a.shape[1]):
        v = float(a[r, c])
        if v <= 0.5:
            continue
        lng, lat = tr * (c + 0.5, r + 0.5)
        cells.append([round(lng, 4), round(lat, 4), round(v, 2)])

json.dump(cells, open(r"E:\economic_wealth_dashboard\ntl_grid.json", "w"))
print(len(cells), "lit cells written to ntl_grid.json")

17052 lit cells written to ntl_grid.json


In [4]:
import rasterio, numpy as np

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    print("pixel size (deg):", src.res)
    print("shape:", a.shape, "=", a.shape[0]*a.shape[1], "pixels")
    print("NaN pixels:", int(np.isnan(a).sum()))
    print("nodata value:", src.nodata)
    print("min:", np.nanmin(a), "max:", np.nanmax(a))
    print("finite pixels > 0.5:", int((np.nan_to_num(a) > 0.5).sum()))

pixel size (deg): (0.004491576420597608, 0.004491576420597608)
shape: (116, 147) = 17052 pixels
NaN pixels: 0
nodata value: None
min: 0.705 max: 86.895004
finite pixels > 0.5: 17052


In [5]:
import rasterio, json, math
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep

shp = gpd.read_file(r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp").to_crs(4326)
district = prep(shp.geometry.union_all())

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    tr = src.transform
    px = abs(src.res[0])
    lng0, lat0 = tr * (0, 0)          # top-left corner of the raster

cells = []
for r in range(a.shape[0]):
    for c in range(a.shape[1]):
        v = float(a[r, c])
        if not math.isfinite(v) or v <= 0.5:
            continue
        lng, lat = tr * (c + 0.5, r + 0.5)
        if not district.contains(Point(lng, lat)):   # keep ONLY inside your boundary
            continue
        cells.append([c, r, round(v, 2)])

out = {"lng0": lng0, "lat0": lat0, "px": px, "cells": cells}
json.dump(out, open(r"E:\economic_wealth_dashboard\ntl_grid.json", "w"))
print(len(cells), "cells inside district | px:", round(px, 6))

8426 cells inside district | px: 0.004492


In [6]:
import rasterio, json, math
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep

shp = gpd.read_file(r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp").to_crs(4326)
district = prep(shp.geometry.union_all())

with rasterio.open(r"E:\economic_wealth_dashboard\lahore_ntl_2025.tif") as src:
    a = src.read(1)
    tr = src.transform
    px = abs(src.res[0])
    lng0, lat0 = tr * (0, 0)

cells = []
for r in range(a.shape[0]):
    for c in range(a.shape[1]):
        v = float(a[r, c])
        if not math.isfinite(v) or v <= 0.5:
            continue
        lng, lat = tr * (c + 0.5, r + 0.5)
        if not district.contains(Point(lng, lat)):
            continue
        cells.append([c, r, round(v, 2)])

out = {"lng0": lng0, "lat0": lat0, "px": px, "cells": cells}
json.dump(out, open(r"E:\economic_wealth_dashboard\ntl_grid.json", "w"))
print(len(cells), "cells inside district")

8426 cells inside district


In [7]:
import osmnx as ox
import geopandas as gpd

shp = gpd.read_file(r"E:\lhr_shp\Lahore_shapefile-20260724T080102Z-1-001\Lahore_shapefile\Lahore.shp").to_crs(4326)
district = shp.geometry.union_all()

tags = {"shop": True, "amenity": True, "office": True,
        "craft": True, "industrial": True, "tourism": True}

pois = ox.features_from_polygon(district, tags)
print("Total POIs:", len(pois))

Total POIs: 6545


In [8]:
print("Shops:", pois["shop"].notna().sum() if "shop" in pois else 0)
print("Amenities:", pois["amenity"].notna().sum() if "amenity" in pois else 0)
print("Offices:", pois["office"].notna().sum() if "office" in pois else 0)

print("\nTop 15 shop types:")
print(pois["shop"].value_counts().head(15))
print("\nTop 15 amenity types:")
print(pois["amenity"].value_counts().head(15))

Shops: 1137
Amenities: 4305
Offices: 567

Top 15 shop types:
shop
bakery              112
clothes              96
yes                  94
supermarket          91
mall                 84
convenience          43
car_repair           39
department_store     39
shoes                36
electronics          34
car                  34
books                22
mobile_phone         21
hairdresser          21
beauty               19
Name: count, dtype: int64

Top 15 amenity types:
amenity
place_of_worship    1110
restaurant           435
bank                 407
school               311
parking              249
hospital             226
fuel                 204
fast_food            142
cafe                  94
grave_yard            87
pharmacy              86
college               84
marketplace           79
university            72
fountain              72
Name: count, dtype: int64


In [9]:
print(pois["amenity"].value_counts().head(30))

amenity
place_of_worship    1110
restaurant           435
bank                 407
school               311
parking              249
hospital             226
fuel                 204
fast_food            142
cafe                  94
grave_yard            87
pharmacy              86
college               84
marketplace           79
university            72
fountain              72
clinic                59
atm                   58
police                54
community_centre      53
bus_station           40
events_venue          38
post_office           30
cinema                29
bench                 22
ice_cream             20
parking_entrance      18
dentist               17
doctors               17
shelter               15
smoking_area          15
Name: count, dtype: int64


In [12]:
keep = pois[["geometry"]].copy()
for col in ["shop", "amenity", "office", "craft", "tourism"]:
    keep[col] = pois[col] if col in pois else None
keep = keep[keep.geometry.notna()].to_crs(4326)
keep["lng"] = keep.geometry.centroid.x
keep["lat"] = keep.geometry.centroid.y
keep.drop(columns="geometry").to_csv(r"E:\economic_wealth_dashboard\lahore_pois.csv", index=False)
print("Saved lahore_pois.csv with", len(keep), "rows")

Saved lahore_pois.csv with 6545 rows


In [13]:
import pandas as pd

df = pd.read_csv(r"E:\economic_wealth_dashboard\lahore_pois.csv")

EXCLUDE = ["place_of_worship", "grave_yard", "fountain", "bench", "shelter",
           "police", "toilets", "drinking_water", "waste_basket", "recycling",
           "smoking_area", "post_box", "telephone", "waste_disposal"]
df = df[~df["amenity"].isin(EXCLUDE)]

SECTOR_MAP = {
  "Retail & Wholesale": {
    "shop": ["*"],
    "amenity": ["marketplace", "vending_machine"]},
  "Food & Hospitality": {
    "amenity": ["restaurant", "cafe", "fast_food", "food_court", "ice_cream", "juice_bar"],
    "tourism": ["hotel", "guest_house", "motel"]},
  "Financial & Professional": {
    "amenity": ["bank", "atm", "bureau_de_change", "money_transfer"],
    "office": ["*"]},
  "Transport & Logistics": {
    "amenity": ["fuel", "parking", "bus_station", "taxi", "car_wash", "car_rental"]},
  "Manufacturing": {
    "craft": ["*"], "industrial": ["*"]},
  "Health & Education": {
    "amenity": ["hospital", "clinic", "pharmacy", "doctors", "dentist",
                "school", "college", "university", "kindergarten", "library"]},
}

def classify(row):
    for sector, rules in SECTOR_MAP.items():
        for col, vals in rules.items():
            v = row.get(col)
            if pd.notna(v) and (vals == ["*"] or v in vals):
                return sector
    return "Other Services"

df["sector"] = df.apply(classify, axis=1)
counts = df["sector"].value_counts()
print("Economic POIs total:", len(df))
print(counts)
print("\nShares (%):")
print((100 * counts / counts.sum()).round(1))
df.to_csv(r"E:\economic_wealth_dashboard\lahore_pois_classified.csv", index=False)

Economic POIs total: 5154
sector
Retail & Wholesale          1218
Financial & Professional    1042
Health & Education           876
Food & Hospitality           870
Other Services               606
Transport & Logistics        511
Manufacturing                 31
Name: count, dtype: int64

Shares (%):
sector
Retail & Wholesale          23.6
Financial & Professional    20.2
Health & Education          17.0
Food & Hospitality          16.9
Other Services              11.8
Transport & Logistics        9.9
Manufacturing                0.6
Name: count, dtype: float64
